In [ ]:
# ---------------------------
# Block 1 — Imports
# ---------------------------

# Standard libs
import os
import time
import math
import json
import random
from typing import List, Tuple, Dict
from collections import defaultdict, Counter

# Scientific / utility
import numpy as np

# Plotting (for later figures)
import matplotlib.pyplot as plt

# Crypto / hashing
import hashlib

# Qiskit (QRNG)
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Optional: Poisson utilities (if you want explicit functions)
from scipy.stats import poisson




-----------------------DECOY STATE BB84 PROTOCOL----------------------

In [2]:
# ---------------------------------------------
# Block 2 — Decoy-State BB84 (with PNS Attack)
# ---------------------------------------------

# BB84 settings (DECOSY)
MU_SIGNAL = 0.5        # mean photon number for signal pulses
MU_DECOY  = 0.1        # mean photon number for decoy pulses
MU_VACUUM = 0.0        # vacuum decoy
BASIS = ['Z', 'X']

# Detection & error settings
DETECTOR_EFFICIENCY = 0.15     # realistic SPAD efficiency
QBER_CHANNEL = 0.02            # 2% physical noise in channel
LOSS_RATE = 0.05               # probability of complete loss

# ---------------------------------------------
# Photon generation using correct Poisson stats
# ---------------------------------------------
def generate_photon_count(mu: float) -> int:
    return np.random.poisson(mu)

# ---------------------------------------------
# Polarization preparation
# ---------------------------------------------
def prepare_polarization(bit: int, basis: str) -> str:
    """
    Returns: one of 'H', 'V', 'D', 'A'
    H/V -> Z basis, D/A -> X basis
    """
    if basis == 'Z':
        return 'H' if bit == 0 else 'V'
    else:
        return 'D' if bit == 0 else 'A'

# ---------------------------------------------
# Measurement function
# ---------------------------------------------
def measure_polarization(state: str, basis: str) -> int:
    """
    If correct basis → deterministic
    If wrong basis → 50% chance
    """
    if basis == 'Z':
        if state in ['H', 'V']:
            return 0 if state == 'H' else 1
        else:
            return np.random.randint(0, 2)
    else:
        if state in ['D', 'A']:
            return 0 if state == 'D' else 1
        else:
            return np.random.randint(0, 2)

# ---------------------------------------------
# PNS Attack Model (optional)
# ---------------------------------------------
def pns_attack(photon_count: int, enable_attack: bool) -> Tuple[int, int]:
    """
    photon_count: original count
    return: (count_arriving_to_bob, eavesdropper_copies)
    """
    if not enable_attack:
        return photon_count, 0

    # Multi-photon pulses only are attacked
    if photon_count > 1:
        eve_keeps = photon_count - 1   # Eve steals extra photons
        return 1, eve_keeps           # Bob gets exactly 1 photon
    else:
        return photon_count, 0

# ---------------------------------------------
# Full BB84 Run (one pulse)
# ---------------------------------------------
def simulate_pulse(enable_pns=False):
    """Simulates a single BB84 pulse"""
    
    # Alice chooses random bit & basis
    a_bit = np.random.randint(0, 2)
    a_basis = np.random.choice(BASIS)

    # Select intensity class (signal / decoy)
    intensity_class = np.random.choice(
        ['signal', 'decoy', 'vacuum'],
        p=[0.8, 0.15, 0.05]
    )
    if intensity_class == 'signal':
        mu = MU_SIGNAL
    elif intensity_class == 'decoy':
        mu = MU_DECOY
    else:
        mu = MU_VACUUM

    # Photon number
    photon_count = generate_photon_count(mu)

    # State preparation
    state = prepare_polarization(a_bit, a_basis)

    # Optional: PNS attack
    photon_count, eve_kept = pns_attack(photon_count, enable_pns)

    # Loss in channel
    if np.random.rand() < LOSS_RATE:
        return None  # Bob gets nothing

    # Detector efficiency
    if photon_count == 0 or np.random.rand() > DETECTOR_EFFICIENCY:
        return None

    # Bob chooses basis
    b_basis = np.random.choice(BASIS)

    # Bob measures
    b_bit = measure_polarization(state, b_basis)

    # Channel noise (QBER)
    if np.random.rand() < QBER_CHANNEL:
        b_bit ^= 1

    return {
        "a_bit": a_bit,
        "a_basis": a_basis,
        "b_bit": b_bit,
        "b_basis": b_basis,
        "class": intensity_class,
        "eve_kept": eve_kept
    }

# ---------------------------------------------
# BB84 Experiment Runner
# ---------------------------------------------
def run_bb84(num_pulses: int, enable_pns=False):
    """
    Returns:
        sifted_key_bits
        decoy_stats
        raw_events
    """

    raw_events = []
    for _ in range(num_pulses):
        ev = simulate_pulse(enable_pns)
        if ev is not None:
            raw_events.append(ev)

    # Sifting
    sifted = []
    decoy_info = []

    for ev in raw_events:
        if ev["a_basis"] == ev["b_basis"]:
            if ev["class"] == 'signal':
                sifted.append(ev["b_bit"])
            else:
                decoy_info.append(ev)

    return sifted, decoy_info, raw_events


---------------------------QRNG----------------------------

In [3]:
# ---------------------------------------------
# Block 3 — QRNG (Qiskit-Aer memory-based, shot-by-shot)
# ---------------------------------------------

from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit, transpile

def qrng_generate_bits(n_bits: int, backend_name: str = 'aer_simulator', use_fixed_seed: bool = False) -> np.ndarray:
    """
    Generate n_bits quantum-random bits using AerSimulator (shot-by-shot memory).
    Returns: numpy array of shape (n_bits,) dtype=np.uint8 with values 0 or 1.

    Parameters:
      - n_bits: number of bits required
      - backend_name: currently unused; kept for API parity (uses AerSimulator)
      - use_fixed_seed: if True, will set a fixed seed in the simulator (useful for debugging)
    """
    backend = AerSimulator()
    max_shots = 8192
    bits_list = []
    remaining = int(n_bits)

    while remaining > 0:
        shots = min(remaining, max_shots)
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)

        compiled = transpile(qc, backend)

        run_kwargs = {"shots": shots, "memory": True}
        if use_fixed_seed:
            run_kwargs["seed_simulator"] = 42  # reproducible debugging

        job = backend.run(compiled, **run_kwargs)
        result = job.result()

        # Prefer per-shot memory (preserves shot order)
        try:
            mem = result.get_memory()
            # mem entries are strings like '0' or '1'
            for m in mem:
                bits_list.append(int(m[-1]))
        except Exception:
            # Fallback: expand counts into a list then shuffle to avoid block runs
            counts = result.get_counts()
            tmp = []
            for bitstr, cnt in counts.items():
                b = int(bitstr[-1])
                tmp.extend([b] * cnt)
            random.shuffle(tmp)
            bits_list.extend(tmp[:shots])

        remaining -= shots

    # Trim and convert to numpy array
    bits_array = np.array(bits_list[:n_bits], dtype=np.uint8)
    return bits_array


def bits_to_hexkey_bytes(bit_array: np.ndarray) -> bytes:
    """
    Pack a numpy bit-array (0/1) into bytes (MSB-first per byte).
    """
    n = len(bit_array)
    pad_len = (8 - (n % 8)) % 8
    if pad_len > 0:
        bit_array = np.concatenate([bit_array, np.zeros(pad_len, dtype=np.uint8)])
    bytes_list = []
    for i in range(0, len(bit_array), 8):
        val = 0
        for j in range(8):
            val = (val << 1) | int(bit_array[i + j])
        bytes_list.append(val)
    return bytes(bytes_list)


def qrng_sanity_check(bit_array: np.ndarray) -> Dict[str, float]:
    """
    Quick sanity metrics for QRNG output. Returns:
      - ones_ratio
      - zeros_ratio
      - approx_min_entropy (per-bit) = -log2(max(p0,p1))
      - small autocorr: fraction of equal adjacent bits (should be near 0.5)
    """
    n = len(bit_array)
    if n == 0:
        return {"ones_ratio": 0.0, "zeros_ratio": 0.0, "min_entropy": 0.0, "adjacent_equal_ratio": 0.0}
    ones = int(np.sum(bit_array))
    zeros = n - ones
    p1 = ones / n
    p0 = zeros / n
    min_entropy = -math.log2(max(p0, p1)) if max(p0, p1) > 0 else 0.0

    # adjacent equal ratio
    if n > 1:
        adjacent_equal = np.sum(bit_array[:-1] == bit_array[1:])
        adjacent_equal_ratio = adjacent_equal / (n - 1)
    else:
        adjacent_equal_ratio = 0.0

    return {
        "ones_ratio": p1,
        "zeros_ratio": p0,
        "min_entropy": min_entropy,
        "adjacent_equal_ratio": adjacent_equal_ratio
    }


# Example usage (sanity):
# bits = qrng_generate_bits(128, use_fixed_seed=False)
# print(qrng_sanity_check(bits))
# print(bits[:64])


---------------------QPP ENCRYPTION WITH DYNAMIC SEEDING-----------------

In [7]:
# =======================
# BLOCK 4 — Quantum Privacy Amplification (QPP)
# =======================

import numpy as np
import os
import hashlib

# ---------- Safe local folder ----------
RESULTS_DIR = "bb84_qrng_qpp_output"
os.makedirs(RESULTS_DIR, exist_ok=True)


# ---------- File / text helpers ----------
def encrypt_text_to_file(plaintext: str, final_key_bits: np.ndarray,
                         out_fname: str = "qpp_cipher.bin") -> str:
    """
    Encrypt a plaintext string using one-time-pad method with final hashed key.
    Writes ciphertext to local RESULTS_DIR/out_fname.
    """
    # Convert plaintext → bytes
    pt_bytes = plaintext.encode("utf-8")

    # Stretch bits to required length
    repeated_key = np.resize(final_key_bits, len(pt_bytes))

    # XOR encryption
    encrypted = bytes([b ^ k for b, k in zip(pt_bytes, repeated_key)])

    out_path = os.path.join(RESULTS_DIR, out_fname)
    with open(out_path, "wb") as f:
        f.write(encrypted)

    return out_path


# ---------- QPP Hash Function ----------
def qpp_hash(raw_key_bits: np.ndarray,
             hash_name: str = "sha256",
             output_bits: int = 128) -> np.ndarray:
    """
    Simulates privacy amplification by hashing.
    """
    # Convert bit-array → byte-string
    bitstring = "".join(str(int(b)) for b in raw_key_bits)
    raw_bytes = int(bitstring, 2).to_bytes((len(raw_key_bits) + 7) // 8, 'big')

    # Hash selection
    if hash_name.lower() == "sha256":
        digest = hashlib.sha256(raw_bytes).digest()
    elif hash_name.lower() == "sha512":
        digest = hashlib.sha512(raw_bytes).digest()
    else:
        raise ValueError("Unsupported hash")

    # Convert digest to bit-array
    digest_bits = np.unpackbits(np.frombuffer(digest, dtype=np.uint8))

    # Reduce to required length
    final_key = digest_bits[:output_bits]

    return final_key


# ---------- QPP Runner ----------
def run_qpp_pipeline(raw_key_bits: np.ndarray,
                     hash_name="sha256",
                     final_bits=128,
                     label="qpp_run") -> dict:
    """
    Runs privacy amplification + stores encrypted sample file.
    """
    final_key = qpp_hash(raw_key_bits, hash_name=hash_name, output_bits=final_bits)

    # Make a demo encryption file
    sample_plaintext = "Quantum-secure encrypted message"
    out_path = encrypt_text_to_file(sample_plaintext, final_key,
                                    out_fname=f"{label}.bin")

    return {
        "final_key_bits": final_key,
        "cipher_file": out_path,
        "hash_used": hash_name,
        "length_bits": len(final_key)
    }


-----------------METRICS--------------

In [15]:
# ---------------------------------------------
# Block 5 — Metrics & Experiment Driver (clean)
# ---------------------------------------------
import json
import time
import math
from collections import Counter

# --------- Remove folders completely ---------
RESULTS_DIR = None   # <- No saving, no directories

# Sample plaintext
SAMPLE_PLAINTEXT = ("AES: Over the years, while certain theoretical vulnerabilities have been "
                    "identified in AES ... (same text)... modern cryptographic applications.")

# --------- Sanity check ----------
required_names = [
    'run_bb84',
    'qrng_generate_bits',
    'expand_key_bits',
    'qpp_encrypt',
    'qpp_decrypt'
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Missing earlier functions: {missing}")

# --------- Metric helpers ----------
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    return sum(bin(x ^ y).count("1") for x, y in zip(a, b))

def metric_encryption_time(plaintext_bytes, key_bits):
    start = time.time()
    ciphertext = qpp_encrypt(plaintext_bytes, key_bits)
    end = time.time()
    return (end - start), ciphertext

def metric_decryption_time(ciphertext, key_bits):
    start = time.time()
    _ = qpp_decrypt(ciphertext, key_bits)
    end = time.time()
    return (end - start)

def metric_avalanche_effect(plaintext_bytes, key_bits):
    original = qpp_encrypt(plaintext_bytes, key_bits)
    modified = bytearray(plaintext_bytes)
    modified[0] ^= 1
    modified_cipher = qpp_encrypt(bytes(modified), key_bits)

    L = min(len(original), len(modified_cipher))
    hd = hamming_distance_bytes(original[:L], modified_cipher[:L])
    total = L * 8
    return hd, hd/total if total > 0 else 0.0

def metric_entropy(data: bytes):
    if not data:
        return 0.0
    freq = Counter(data)
    N = len(data)
    return -sum((c/N) * math.log2(c/N) for c in freq.values())

def flip_bit(arr, idx):
    new = arr.copy()
    new[idx] ^= 1
    return new

def metric_key_sensitivity(plaintext_bytes, key_bits):
    original = qpp_encrypt(plaintext_bytes, key_bits)
    modified_key = flip_bit(key_bits, 0)
    modified = qpp_encrypt(plaintext_bytes, modified_key)

    L = min(len(original), len(modified))
    hd = hamming_distance_bytes(original[:L], modified[:L])
    total = L * 8
    return hd, hd/total if total > 0 else 0.0

def metric_frequency_test(ciphertext):
    bitstring = "".join(f"{b:08b}" for b in ciphertext)
    ones = bitstring.count("1")
    total = len(bitstring)
    return ones/total, 1 - (ones/total)

def metric_permutation_uniformity(key_bits):
    session_bits = expand_key_bits(key_bits, target_bits=128)
    block_size = max(1, len(session_bits)//8)
    return True, list(range(block_size))[:32]

# --------- Final key derivation ----------
def derive_final_key_from_sifted(sifted_bits, target_bits=128):
    if not sifted_bits:
        return np.zeros(target_bits, dtype=np.uint8)

    bb84_arr = np.array(sifted_bits, dtype=np.uint8)
    qrng_arr = np.array(
        [int(b) for b in qrng_generate_bits(len(bb84_arr), False)],
        dtype=np.uint8
    )

    final = np.bitwise_xor(bb84_arr, qrng_arr)
    return expand_key_bits(final, target_bits=target_bits)

# --------- Experiment runner ----------
def run_experiment_with_metrics(num_pulses=20000, enable_pns=False,
                                target_bits=128, max_tries=3, label="exp"):

    curr = num_pulses
    for attempt in range(1, max_tries+1):
        print(f"\n[{label}] Attempt {attempt}/{max_tries} — N={curr}, PNS={'ON' if enable_pns else 'OFF'}")

        sifted, _, events = run_bb84(curr, enable_pns=enable_pns)
        print(f"[{label}] Sifted bits: {len(sifted)}")

        key = derive_final_key_from_sifted(sifted, target_bits)
        print(f"[{label}] Final key bits: {len(key)}")

        if len(key) >= target_bits:
            pt = SAMPLE_PLAINTEXT.encode()

            enc_t, ct = metric_encryption_time(pt, key)
            dec_t = metric_decryption_time(ct, key)
            av_hd, av_ratio = metric_avalanche_effect(pt, key)
            entropy = metric_entropy(ct)
            ks_hd, ks_ratio = metric_key_sensitivity(pt, key)
            ones, zeros = metric_frequency_test(ct)
            perm_ok, perm_sample = metric_permutation_uniformity(key)

            metrics = {
                "label": label,
                "pulses": curr,
                "sifted_count": len(sifted),
                "final_key_bits_len": len(key),
                "enc_time_s": enc_t,
                "dec_time_s": dec_t,
                "avalanche_hd": av_hd,
                "avalanche_ratio": av_ratio,
                "ciphertext_entropy": entropy,
                "key_sensitivity_hd": ks_hd,
                "key_sensitivity_ratio": ks_ratio,
                "freq_ones": ones,
                "freq_zeros": zeros,
                "permutation_uniform": perm_ok,
                "permutation_sample_prefix": perm_sample
            }

            return key, metrics, events

        print(f"[{label}] insufficient key bits; increasing pulses to {curr*2}")
        curr *= 2

    print(f"[{label}] Exhausted tries — returning best effort")
    return np.zeros(target_bits, dtype=np.uint8), {
        "label": label,
        "pulses": curr,
        "note": "failed to obtain key"
    }, []

# --------- Run baseline + PNS ----------
print("\n=== Running baseline ===")
final_baseline, metrics_baseline, events_baseline = run_experiment_with_metrics(
    num_pulses=20000, enable_pns=False, target_bits=128, max_tries=3, label="baseline"
)

print("\n=== Running PNS stress ===")
final_pns, metrics_pns, events_pns = run_experiment_with_metrics(
    num_pulses=20000, enable_pns=True, target_bits=128, max_tries=3, label="pns_stress"
)

# --------- Compact summary ----------
print("\n=== Summary ===")
print("Baseline:", {k: metrics_baseline.get(k) for k in [
    "sifted_count", "final_key_bits_len", "avalanche_ratio", "ciphertext_entropy"]})
print("PNS:", {k: metrics_pns.get(k) for k in [
    "sifted_count", "final_key_bits_len", "avalanche_ratio", "ciphertext_entropy"]})



=== Running baseline ===

[baseline] Attempt 1/3 — N=20000, PNS=OFF
[baseline] Sifted bits: 423
[baseline] Final key bits: 128

=== Running PNS stress ===

[pns_stress] Attempt 1/3 — N=20000, PNS=ON
[pns_stress] Sifted bits: 444
[pns_stress] Final key bits: 128

=== Summary ===
Baseline: {'sifted_count': 423, 'final_key_bits_len': 128, 'avalanche_ratio': 0.3342013888888889, 'ciphertext_entropy': 4.405747876710716}
PNS: {'sifted_count': 444, 'final_key_bits_len': 128, 'avalanche_ratio': 0.3220486111111111, 'ciphertext_entropy': 4.405747876710716}
